This notebook shows the local execution of Comp2Prot/Prot2Comp and of the individual
database classes.

NOTE: Before running this: generate your local data folder with: \
`python -m cpiextract.utils.prepare_cpiextract_data --data-path <your path>` \
or follow the db_preprocessing notebook.

In [ ]:
#General packages
import pandas as pd
from tqdm import tqdm
import os
from cpiextract import Comp2Prot
from cpiextract import Prot2Comp
from cpiextract.utils import load_dbs, load_pubchem_files, check_status, compound_identifiers
import time
import cProfile
import numpy as np

import glob
import re
from pathlib import Path
from rdkit import Chem
from rdkit.Chem.inchi import MolFromInchi
from rdkit.Chem import Descriptors, rdMolDescriptors
import pubchempy as pcp

In [ ]:
# Root data path - point this at your own prepared CPIExtract data folder
data_path = '/path/to/CPE_data/'

## Load in Datasets and Pipelines

In [ ]:
# Readiness report across all 9 databases before loading
check_status(data_path)

# Loads every standardized CSV present into the {key: DataFrame} dict Comp2Prot/Prot2Comp expect.
# Missing databases are skipped with a warning rather than raising.
dbs = load_dbs(data_path)

# Returns {'db_file': path} if a local PubChem duckdb build is present, otherwise None -
# in which case PubChem falls back to the live PUG-REST API instead.
pubchem_files = load_pubchem_files(data_path)

# Establish Pipelines
C2P = Comp2Prot(execution_mode='local', dbs=dbs, pubchem_files=pubchem_files, server_select='mygene')
P2C = Prot2Comp(execution_mode='local', dbs=dbs, pubchem_files=pubchem_files, server_select='mygene')

## Load in Compound List

Any CSV with an `inchikey` column works here.

DrugBank compounds

In [ ]:
chem_dat = pd.read_csv(data_path + 'db_compounds.csv')
output_path = 'data/output/C2P/'
os.makedirs(output_path, exist_ok=True)

FooDB compounds

In [ ]:
chem_dat = pd.read_csv(data_path + 'foodb_compounds.csv')
output_path = 'data/output/foodb/'
os.makedirs(output_path, exist_ok=True)

## Run Comp2Prot on Compound List

In [ ]:
# Setup checkpoint directory
checkpoint_dir = os.path.join(output_path, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

### Run with PubChem compound information

In [ ]:
c2p_agg = pd.DataFrame()
c2p_fail = pd.DataFrame()
states = pd.DataFrame()

r = 0
for h in tqdm(range(0, 10)):

    try:
        inputid = chem_dat['inchikey'].iloc[h]
    except (KeyError, IndexError):
        continue

    try:
        # Every database is searched for all stereoisomers of the input structure
        [comp_agg, state, comp_raw] = C2P.comp_interactions(inputid,pchembl_grouping='combined',verbose=False)
        states = pd.concat([states, state])
        c2p_agg = pd.concat([c2p_agg, comp_agg])
        #print(f'{h} done')
    except Exception as e:
        print(f'{h} failed: {e}')
        c2p_fail.loc[r, "failed_id"] = inputid  # Collects failed input ids
        r = r + 1

    # Saves a checkpoint file periodically, in case a long run gets interrupted
    checkpoint_every = 50
    if h % checkpoint_every == 0:
        filename = os.path.join(checkpoint_dir, f'iter_{h}.csv')
        c2p_agg.to_csv(filename, sep=',', index=False)

# Save completed dataframe to file
filename = os.path.join(output_path, 'c2p_pubchem_compounds.csv')
c2p_agg.to_csv(filename, sep=',', index=False)

### Run with rdkit compound information

Use this when you have compounds (e.g. a novel/hypothetical structure) that aren't
necessarily in PubChem at all. \
`prebuilt_comp_ids` allows users to query compounds not in PubChem. \
This expects a full InChI string per compound (not an InChIKey) - adjust `'inchi'` below to whichever
column in your own compound list.

In [ ]:
def compound_from_inchi(input_id):
    mol = Chem.MolFromInchi(input_id)
    if mol is None:
        raise ValueError(f"RDKit could not parse InChI: {input_id!r}")

    inchikey = Chem.MolToInchiKey(mol)
    row = {
        'input_id': inchikey,
        'CID': None,
        'molecular_formula': rdMolDescriptors.CalcMolFormula(mol),
        'molecular_weight': Descriptors.MolWt(mol),
        'smiles': Chem.MolToSmiles(mol, isomericSmiles=True),
        'connectivity_smiles': Chem.MolToSmiles(mol, isomericSmiles=False),
        'inchi': Chem.MolToInchi(mol),
        'inchikey': inchikey,
        'iupac_name': None,
        'synonyms': [[]],
        'inchikey_fb': inchikey.split('-')[0],
    }
    return row

In [ ]:
# 'inchi' here should be a column of full InChI strings in your own compound list -
# NOT the same as the 'inchikey' column used in the PubChem-based approach above.
rdkit_dat = chem_dat['inchi'].apply(lambda x: pd.Series(compound_from_inchi(x)))

In [ ]:
c2p_agg = pd.DataFrame()
c2p_fail = pd.DataFrame()
states = pd.DataFrame()

r = 0
for h in tqdm(range(0, 10)):

    try:
        inputid = rdkit_dat['inchikey'].iloc[h]
        comp_ids = rdkit_dat.iloc[h]
    except (KeyError, IndexError):
        continue

    try:
        [comp_agg,state,comp_raw] = C2P.comp_interactions(inputid, prebuilt_comp_ids=comp_ids,
                                                            pchembl_grouping='combined',verbose=False)
        states = pd.concat([states, state])
        c2p_agg = pd.concat([c2p_agg, comp_agg])
        #print(f'{h} done')
    except Exception as e:
        print(f'{h} failed: {e}')
        c2p_fail.loc[r, "failed_id"] = inputid
        r = r + 1

    checkpoint_every = 50
    if h % checkpoint_every == 0:
        filename = os.path.join(checkpoint_dir, f'rdkit_iter_{h}.csv')
        c2p_agg.to_csv(filename, sep=',', index=False)

# Save completed dataframe to file
filename = os.path.join(output_path, 'c2p_rdkit_compounds.csv')
c2p_agg.to_csv(filename, sep=',', index=False)

### Combining checkpoint files

If a long run gets interrupted, this reassembles the full result from whatever
checkpoint files were saved along the way.

In [ ]:
pattern = str(Path(checkpoint_dir) / "iter_*.csv")
files = sorted(glob.glob(pattern), key=lambda f: int(re.search(r"iter_(\d+)\.csv", f).group(1)))
print(f"Found {len(files)} checkpoint files")

if files:
    combined = pd.concat((pd.read_csv(f) for f in files), ignore_index=True).drop_duplicates()
    print("Combined size:", combined.shape)
    combined.to_csv(os.path.join(output_path, 'c2p_combined.csv'), index=False)

## Run Comp2Prot on Single Compound

In [ ]:
inputid = 'SUVMJBTUFCVSAD-UHFFFAOYSA-N'

In [ ]:
[comp_agg, states, comp_raw] = C2P.comp_interactions(input_id=inputid,pchembl_grouping='combined',verbose=False)

### Test time required

In [ ]:
cProfile.run("C2P.comp_interactions(inputid, verbose=False)")

## Comp2Prot with select Databases

In [ ]:
# The second argument is a string listing the databases needed.
# An underscore separates the database keys if using more than one.
[comp_agg, states, comp_raw] = C2P.comp_interactions_select(inputid,'ctd_stitch',
                                                            pchembl_grouping='combined',verbose=False)

### Run each database separately

Each database class's own `compounds()`/`proteins()` methods can be used directly,
without going through the Comp2Prot/Prot2Comp orchestration - useful for debugging a
single source or inspecting its raw output. `dbs` here is the same dict loaded via
`load_dbs()` above; PubChem is constructed differently from the rest, since it uses a
local DuckDB file (or falls back to the live API) rather than a DataFrame.

In [ ]:
from cpiextract.databases import *

inputid = 'SUVMJBTUFCVSAD-UHFFFAOYSA-N'
comp_ids = compound_identifiers(inputid,verbose=True)

##### Run PubChem only

In [ ]:
pc = PubChem(bioact_file=(pubchem_files or {}).get('db_file'))
[pc_dat, pc_state, pc_raw] = pc.compounds(input_comp=comp_ids,pchembl_grouping='combined',verbose=False)

##### Run ChEMBL only

In [ ]:
chembl = ChEMBL(database=dbs['chembl'])
[chembl_dat, chembl_state, chembl_raw] = chembl.compounds(input_comp=comp_ids,pchembl_grouping='combined')

##### Run BindingDB only

In [ ]:
bdb = BindingDB(database=dbs['bdb'])
[bdb_dat, bdb_state, bdb_raw] = bdb.compounds(input_comp=comp_ids,pchembl_grouping='combined')

##### Run STITCH only

In [ ]:
stitch = Stitch(database=dbs['stitch'])
[stitch_dat, stitch_state, stitch_raw] = stitch.compounds(input_comp=comp_ids,pchembl_grouping='combined',
                                                          experimental_thres=400)

##### Run CTD only

In [ ]:
ctd = CTD(database=dbs['ctd'])
[ctd_dat, ctd_state, ctd_raw] = ctd.compounds(input_comp=comp_ids,pchembl_grouping='combined')

##### Run DTC only

In [ ]:
dtc = DTC(database=dbs['dtc'])
[dtc_dat, dtc_state, dtc_raw] = dtc.compounds(input_comp=comp_ids,pchembl_grouping='combined',verbose=False)

##### Run OTP only

In [ ]:
otp = OTP(database=dbs['otp'])
[otp_dat, otp_state, otp_raw] = otp.compounds(input_comp=comp_ids,pchembl_grouping='combined')

##### Run DrugCentral only

In [ ]:
dc = DrugCentral(database=dbs['dc'])
[dc_dat, dc_state, dc_raw] = dc.compounds(input_comp=comp_ids,pchembl_grouping='combined')

##### Run DrugBank only

In [ ]:
db = DB(database=dbs['db'])
[db_dat, db_state, db_raw] = db.compounds(input_comp=comp_ids,pchembl_grouping='combined')

## Example of Prot2Comp

Expects a CSV with an HGNC ID column (values like `HGNC:3535`).

In [ ]:
inputids = pd.read_csv(data_path + 'db_proteins.csv')
output_path = 'data/output/P2C/'
os.makedirs(output_path, exist_ok=True)

In [ ]:
p2c_agg = pd.DataFrame()
p2c_fail = pd.DataFrame()
states = pd.DataFrame()

r = 0
for h in tqdm(range(0, 5)):
    protid = inputids['HGNC ID'].iloc[h]

    try:
        [prot_agg, state, prot_raw] = P2C.prot_interactions(input_id=protid,pchembl_grouping='combined',
                                                            verbose=False)
        p2c_agg = pd.concat([p2c_agg, prot_agg])
        states = pd.concat([states, state])
        #print(f'{h} done')
    except Exception as e:
        print(f'{h} failed: {e}')
        p2c_fail.loc[r, "failed_id"] = protid
        r = r + 1

    checkpoint_every = 50
    if h % checkpoint_every == 0:
        filename = os.path.join(output_path, f'p2c_iter_{h}.csv')
        p2c_agg.to_csv(filename, sep=',', index=False)

# Save completed dataframe to file
filename = os.path.join(output_path, 'p2c_results.csv')
p2c_agg.to_csv(filename, sep=',', index=False)